# MP31 — Transformer-Based SR Baseline

[Brief](README.md) · [Validation](VALIDATION.md). Prerequisite: MP30.

Use already executed CNN/window-Transformer development runs to assess a compact baseline honestly. This notebook does not retrain, download checkpoints, time benchmarks or repeat the frozen test. A tiny untrained synthetic shape check is separate from recorded accuracy.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import torch
import matplotlib.pyplot as plt
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
sys.path.insert(0,str(root/'stage_07_transformers/MP31_transformer_based_sr_baseline'))
from benchmark_tools import comparison
from rssr.models import SRModel
report=comparison(root)
assert len(report['rows'])==6


## 1. Architecture and tensor contract

The existing educational model maps[batch,4,H,W] RGBNIR input through a3×3 convolution to width16; an unshifted4×4-token window attention block adds local context; a3×3 tail creates16 channels and PixelShuffle×2 restores four bands at[batch,4,2H,2W]. A bicubic skip supplies the base image. Replicated boundary padding is removed after window reconstruction. Windows have two attention heads and learned position parameters.

This is inspired by window-based restoration but lacks faithful shifted-window stacks and residual Swin Transformer groups. Official SwinIR is a separately recorded RGB/DIV2K-pretrained comparison, not this compact model.

In [ ]:
torch.manual_seed(31);torch.set_num_threads(1)
model=SRModel(bands=4,width=16,kind='transformer').eval()
with torch.no_grad():output=model(torch.zeros(1,4,7,9))
assert output.shape==(1,4,14,18) and torch.isfinite(output).all()
print('Synthetic shape check:',tuple(output.shape),'parameters',sum(p.numel() for p in model.parameters()))


## 2. Common metric and budget protocol

Four training/four validation NAIP aerial scenes,16 fixed patches per scene, RGBNIR, DN/10000 (not calibrated reflectance). Synthetic x2 LR: Gaussian σ0.8 then area downsampling. Metrics use range1,border2 and no clipping or luminance conversion. Aggregate tiles within each scene, then equal-weight scenes.

Both model families used seeds17,23,41, Adam lr0.001, batch4, L1 loss,1000 updates and minimum validation L1 selection every100 updates. Same updates do not equalize parameter count, training time or compute. This teaching export reads validation evidence only.

PSNR=10log10(range²/MSE). For range1,MSE0.01 the answer is20dB. Averaging per-scene PSNR is different from computing PSNR from pooled MSE; retain the source convention.

In [ ]:
assert np.isclose(10*np.log10(1/.01),20)
for row in report['rows']:
    print(row['model'],row['seed'],'PSNR',round(row['model_psnr'],4),
          'bicubic',round(row['bicubic_psnr'],4),'delta',round(row['delta_psnr'],4),
          'run',row['run_id'])


## 3. Baseline comparison including an unfavorable result

Across the three seeds the CNN's mean validation gain over bicubic is1.570925dB; the compact Transformer gains0.618003dB. Their seed sample SDs are0.281768 and0.246293dB. These are training variability over three seeds, not geographic confidence intervals. The experiment does not support Transformer superiority.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for i,summary in enumerate(report['summary']):
    values=[r['delta_psnr'] for r in report['rows'] if r['model']==summary['model']]
    ax.scatter([i]*len(values),values,color='#176e8b')
    ax.errorbar(i,summary['mean_delta_db'],yerr=summary['seed_sd_delta_db'],fmt='s',color='#d96724',capsize=6)
ax.set(xticks=[0,1],xticklabels=['CNN','Educational window Transformer'],ylabel='Validation PSNR gain over bicubic (dB)',title='Dots: seeds; square ± sample seed SD; four validation scenes')
ax.axhline(0,color='gray',lw=1);fig.tight_layout();plt.show()


## 4. Parameters are not runtime

CNN has7552 parameters; compact Transformer5392. Read the separately measured CPU table below. It used float32, input[1,4,64,64], five warmups and twenty repetitions on its recorded machine/thread count. These times were not measured in this notebook. Sampled process RSS is not allocator peak. Partial MAC counts exclude normalization, activations, pooling, softmax, interpolation and elementwise work.

In [ ]:
print(report['benchmark_environment'])
for row in report['efficiency']:
    print(row['variant'],'parameters',row['parameters'],'median seconds',row['median_seconds'],
          'p95',row['p95_seconds'],'sampled RSS',row['sampled_process_peak_rss_bytes'],
          'partial MACs',row['conv_linear_attention_macs'])


## 5. Failure analysis and next experiment

The compact Transformer underperforms this CNN under the declared budget despite fewer parameters. Possible explanations include insufficient updates, architecture capacity, window design or this small dataset; the present comparison does not isolate their causes. A fair next experiment would predeclare a single change and evaluate on development data, leaving the frozen test untouched.

The official pretrained SwinIR result cannot be pooled into this four-band equal-update table: it uses RGB, natural-image DIV2K pretraining and different degradation/budget. Small synthetic validation gains do not establish real satellite cross-sensor validity. Negative findings remain visible.

## Practice and separate solutions

Compute the gap between mean validation gains; explain why7552 versus5392 cannot predict latency; distinguish seed SD from geographic uncertainty. Solutions:0.952921dB favoring the CNN in this protocol; runtime also depends on operations, memory access and implementation; three training seeds do not sample new geographic regions.

Reflection: benchmarking requires shared metrics and explicit unequal budgets, not architecture labels. Next: explore satellite imagery in stage08, then combine branches in stage09 while preserving these baselines. Source data and artifact hashes remain in the reports. No new training/held-out evaluation or learner assessment occurred.

References: [Swin Transformer](https://arxiv.org/abs/2103.14030), [SwinIR](https://arxiv.org/abs/2108.10257). The shared educational implementation remains distinct from the official model.

In [ ]:
gap=report['summary'][0]['mean_delta_db']-report['summary'][1]['mean_delta_db']
assert np.isclose(gap,.9529214663987762)
assert all(r['config']['steps']==1000 and r['config']['batch_size']==4 for r in report['rows'])
print('Recorded development gap:',gap,'dB; not a held-out conclusion')
